### Notebook to graph and test Results (Draft)

In [1]:
import json
from pathlib import Path
import pandas as pd

import numpy as np 


In [2]:
def parse_run(filepath: Path) -> dict:
    """Parsea un archivo JSON individual de resultados GA."""
    with open(filepath, "r") as f:
        data = json.load(f)

    algo = filepath.parent.name  # flat o joint

    return {
        "algo": algo,
        "env_id": data["meta"]["env_id"],
        "seed": data["meta"]["seed"],
        "runtime_sec": data["meta"]["runtime_sec"],
        "evals_total": data["meta"]["budget"]["evals_total"],
        "U_best": data["best_utility"],
        "curve_best": data["curves"]["best"],
        "curve_mean": data["curves"]["mean"],
        "evals_cum": data["meta"]["evals_cum"],
        "best_by_selectors": data.get("best_by_selectors", []),
        "pool_sizes": data["meta"].get("pool_sizes"),
        "sizes": data["meta"].get("sizes"),
        "alphabet": data["meta"].get("alphabet"),
    }

def collect_results(root_dir: Path) -> pd.DataFrame:
    """Recorre las carpetas de resultados y consolida todas las corridas."""
    records = []
    for algo_dir in ["flat", "joint"]:
        path = root_dir / algo_dir
        if not path.exists():
            print(f"⚠️  No se encontró carpeta {algo_dir}, se omite.")
            continue
        for f in path.glob("*.json"):
            try:
                records.append(parse_run(f))
            except Exception as e:
                print(f"❌ Error en {f}: {e}")
    return pd.DataFrame(records)

In [3]:
# --- Métricas estructurales ---

def compute_scr(row):
    if not row["best_by_selectors"]:
        return np.nan
    visited = len(set([str(s) for s, _ in row["best_by_selectors"]]))
    total = np.prod(row["pool_sizes"]) if row["pool_sizes"] else np.nan
    return visited / total if total else np.nan

def gini(x):
    if len(x) == 0 or np.sum(x) == 0:
        return 0
    x = np.sort(np.abs(x))
    n = len(x)
    index = np.arange(1, n + 1)
    return ((2 * index - n - 1) * x).sum() / (n * x.sum())

def compute_sfi(row):
    if not row["best_by_selectors"]:
        return np.nan
    counts = pd.Series([str(s) for s, _ in row["best_by_selectors"]]).value_counts().values
    return 1 - gini(counts)

def compute_rbs(row):
    try:
        num = np.prod([pow(a, k) for a, k in zip(row["pool_sizes"], row["sizes"])])
        den = np.prod([math.comb(a + k - 1, k) for a, k in zip(row["alphabet"], row["sizes"])])
        return num / den
    except Exception:
        return np.nan

# --- Métricas de desempeño ---

def compute_time_to_best(row):
    best_val = row["U_best"]
    curve = row["curve_best"]
    for i, v in enumerate(curve):
        if v == best_val:
            return row["evals_cum"][i]
    return row["evals_cum"][-1]

def compute_diversity(curve):
    """Dispersión de la media poblacional (proxy de diversidad)."""
    curve = np.array(curve)
    return np.std(curve)

def compute_convergence_slope(curve):
    """Pendiente de mejora en primeras generaciones."""
    k = max(3, int(0.2 * len(curve)))
    y = np.array(curve[:k])
    x = np.arange(len(y))
    slope, _ = np.polyfit(x, y, 1)
    return slope

In [9]:


# === 1. Definir rutas ===
root_results = Path(".")
env_index_path = Path("../../data/env_index.csv")
output_csv = Path("./aggregates/results_df.csv")

print("📂 Cargando resultados desde:", root_results.resolve())

# === 2. Cargar y combinar resultados ===
df_runs = collect_results(root_results)
print(f"✅ {len(df_runs)} corridas cargadas ({df_runs['algo'].unique().tolist()})")



📂 Cargando resultados desde: C:\Users\andre\Repositories\FTZ_model_2.0\simulations\ga_benchmark\results\FTZ_EvoBench_Test__20251011_095406
✅ 592 corridas cargadas (['flat', 'joint'])


In [16]:
df_runs.head(5)

,algo,env_id,seed,runtime_sec,evals_total,U_best,curve_best,curve_mean,evals_cum,best_by_selectors,pool_sizes,sizes,alphabet
0,flat,g10_high_0,11,81.242489,1800,229.68,"[42.05, 50.95, 61.18, 73.37, 80.07, 89.01, 100...","[-110.8336, -18.720399999999998, 27.2952000000...","[50, 85, 120, 155, 190, 225, 260, 295, 330, 36...",[],None,None,None
1,flat,g10_high_0,17,80.039593,1800,224.40,"[47.05, 47.05, 56.99, 68.3, 72.35, 87.41, 100....","[-115.87580000000001, -41.0074, 1.705399999999...","[50, 85, 120, 155, 190, 225, 260, 295, 330, 36...",[],None,None,None
2,flat,g10_high_0,23,81.074286,1800,228.63,"[20.96, 36.56, 52.0, 83.04, 83.04, 103.07, 103...","[-130.9716, -29.103, 4.8671999999999995, 23.65...","[50, 85, 120, 155, 190, 225, 260, 295, 330, 36...",[],None,None,None
3,flat,g10_high_0,42,80.857561,1800,225.46,"[52.94, 67.07, 70.88, 73.72, 103.79, 109.35, 1...","[-113.4728, -20.2364, 19.0588, 40.157799999999...","[50, 85, 120, 155, 190, 225, 260, 295, 330, 36...",[],None,None,None
4,flat,g10_high_0,73,80.700566,1800,228.63,"[72.5, 75.8, 95.31, 95.31, 108.05, 108.89, 121...","[-107.1884, 7.989799999999998, 37.500600000000...","[50, 85, 120, 155, 190, 225, 260, 295, 330, 36...",[],None,None,None


In [14]:
# === 3. Cargar metadata de entornos ===
env_meta = pd.read_csv(env_index_path)




In [17]:
env_meta.rename(columns={"graph_id": "env_id"}, inplace=True)

In [18]:
env_meta.head()

,env_id,N,density,max_depth,price_file
0,g6_low_0,6,0.25,3,g6_low_0_prices.json
1,g6_low_1,6,0.25,3,g6_low_1_prices.json
2,g6_low_2,6,0.25,3,g6_low_2_prices.json
3,g6_medium_0,6,0.50,5,g6_medium_0_prices.json
4,g6_medium_1,6,0.50,5,g6_medium_1_prices.json


In [19]:
df = df_runs.merge(env_meta, on="env_id", how="left")

In [20]:
# === 4. Calcular métricas ===
df["SCR"] = df.apply(compute_scr, axis=1)
df["SFI"] = df.apply(compute_sfi, axis=1)
df["RBS"] = df.apply(compute_rbs, axis=1)
df["time_to_best"] = df.apply(compute_time_to_best, axis=1)
df["diversity_initial"] = df["curve_mean"].apply(lambda c: np.std(c[:5]))
df["diversity_final"] = df["curve_mean"].apply(lambda c: np.std(c[-5:]))
df["delta_diversity"] = df["diversity_final"] - df["diversity_initial"]
df["convergence_slope"] = df["curve_best"].apply(compute_convergence_slope)
df["runtime_efficiency"] = df["U_best"] / df["runtime_sec"]

# === 5. Guardar CSV ===
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)
print(f"💾 CSV guardado en: {output_csv.resolve()}")

# === 6. Mostrar resumen rápido ===
summary = (
    df.groupby(["algo"])
      .agg({"U_best": ["mean", "std"], "runtime_sec": "mean", "SCR": "mean"})
)
print("\n📊 Resumen global por algoritmo:\n", summary)


💾 CSV guardado en: C:\Users\andre\Repositories\FTZ_model_2.0\simulations\ga_benchmark\results\FTZ_EvoBench_Test__20251011_095406\aggregates\results_df.csv

📊 Resumen global por algoritmo:
             U_best              runtime_sec       SCR
              mean          std        mean      mean
algo                                                 
flat  -1621.290845  3875.973345  342.535422       NaN
joint -1529.751520  3598.287625  360.533879  0.542208
